# Analytics Pipeline Latency Report

**Author:** Aarya Parekh · IIT Bombay  
**Project:** Real-Time Market Microstructure Analyzer

This report profiles the per-tick computation latency of each analytics module in the pipeline. Understanding where compute time is spent is critical for a real-time system — if processing a tick takes longer than the tick interval (200ms for our mock source), the system falls behind and data queues up.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Style
SURFACE = '#fcfcfb'
TEXT_PRIMARY = '#0b0b0b'
TEXT_SECONDARY = '#52514e'
TEXT_MUTED = '#9e9d97'
GRID_COLOR = '#e8e8e4'

# Module colors (categorical palette, slot order)
MODULE_COLORS = {
    'spread_us': '#2a78d6',
    'ofi_us': '#eb6834',
    'vwap_us': '#1baf7a',
    'volume_us': '#eda100',
    'kyle_us': '#7551c2',
    'amihud_us': '#37a6a6',
    'roll_us': '#c76b27',
    'trade_quote_us': '#6b7f3a',
    'anomaly_us': '#e87ba4',
    'overhead_us': '#9e9d97',
}
MODULE_LABELS = {
    'spread_us': 'Spread',
    'ofi_us': 'OFI',
    'vwap_us': 'VWAP',
    'volume_us': 'Volume',
    'kyle_us': 'Kyle Lambda',
    'amihud_us': 'Amihud',
    'roll_us': 'Roll Spread',
    'trade_quote_us': 'Trade/Quote Var',
    'anomaly_us': 'Anomaly Det.',
    'overhead_us': 'Overhead',
}

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'axes.edgecolor': GRID_COLOR, 'axes.labelcolor': TEXT_SECONDARY,
    'axes.grid': True, 'grid.color': GRID_COLOR, 'grid.linewidth': 0.5,
    'xtick.color': TEXT_MUTED, 'ytick.color': TEXT_MUTED,
    'text.color': TEXT_PRIMARY, 'font.size': 10,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 120, 'legend.frameon': False,
})

# Load data
df = pd.read_csv('../data/latency_timings.csv')
with open('../data/latency_summary.json') as f:
    summary = json.load(f)

print(f'Ticks profiled: {len(df):,}')
print(f'Mean total latency: {summary["total"]["mean_us"]:.1f} μs')
print(f'P99 total latency: {summary["total"]["p99_us"]:.1f} μs')
print(f'Throughput: {1_000_000 / summary["total"]["mean_us"]:,.0f} ticks/sec')

---
## 1. Headline Numbers

With a 200ms tick interval, the analytics pipeline uses less than 1% of the available compute budget per tick. The system has >99% headroom for additional modules or higher tick rates.

In [ ]:
# Stat tiles
fig, axes = plt.subplots(1, 4, figsize=(14, 2.5))

stats_data = [
    ('Mean Latency', f'{summary["total"]["mean_us"]:.0f} μs', '#2a78d6'),
    ('P99 Latency', f'{summary["total"]["p99_us"]:.0f} μs', '#eb6834'),
    ('Throughput', f'{1_000_000 / summary["total"]["mean_us"]:,.0f} tps', '#1baf7a'),
    ('Budget Used', f'{summary["total"]["mean_us"] / 200_000 * 100:.2f}%', '#eda100'),
]

for ax, (label, value, color) in zip(axes, stats_data):
    ax.text(0.5, 0.55, value, transform=ax.transAxes,
            ha='center', va='center', fontsize=22, fontweight='bold', color=color)
    ax.text(0.5, 0.15, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color=TEXT_SECONDARY)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 2. Time Breakdown by Module

Where does the compute time actually go? This is the most important chart — it reveals which module to optimise if latency ever becomes a constraint.

In [ ]:
# Horizontal bar — mean latency per module
fig, ax = plt.subplots(figsize=(10, 4))

modules = list(MODULE_LABELS.keys())
means = [df[m].mean() for m in modules]
colors = [MODULE_COLORS[m] for m in modules]
labels = [MODULE_LABELS[m] for m in modules]

# Sort by value
order = np.argsort(means)
bars = ax.barh([labels[i] for i in order], [means[i] for i in order],
               color=[colors[i] for i in order], height=0.5)

for bar, idx in zip(bars, order):
    val = means[idx]
    pct = summary['mean_breakdown_pct'].get(
        list(summary['mean_breakdown_pct'].keys())[idx], 0)
    ax.text(val + max(means)*0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} μs ({pct:.1f}%)', va='center', fontsize=9, color=TEXT_SECONDARY)

ax.set_xlabel('Mean Latency (μs)')
ax.set_title('Per-Module Mean Latency')
plt.tight_layout()
plt.show()

---
## 3. Latency Distribution

The tail matters more than the mean. In real-time systems, a P99 spike that exceeds the tick budget causes data loss. Here we check the total latency distribution and per-module tails.

In [ ]:
# Total latency histogram
fig, ax = plt.subplots(figsize=(12, 4))

ax.hist(df['total_us'], bins=80, color='#2a78d6', alpha=0.8,
        edgecolor='white', linewidth=0.3)

# Percentile markers
for pct, style in [(50, '--'), (95, '-.'), (99, ':')]:
    val = np.percentile(df['total_us'], pct)
    ax.axvline(val, color='#e34948', linewidth=1, linestyle=style)
    ax.text(val, ax.get_ylim()[1]*0.9, f'P{pct}: {val:.0f}μs',
            fontsize=8, color='#e34948', ha='left', va='top', rotation=0)

ax.set_title('Total Per-Tick Latency Distribution')
ax.set_xlabel('Latency (μs)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Per-module percentile comparison
fig, ax = plt.subplots(figsize=(10, 5))

modules_no_overhead = [m for m in modules if m != 'overhead_us']
x = np.arange(len(modules_no_overhead))
w = 0.18

for j, (pct, offset) in enumerate([(50, -1.5), (95, -0.5), (99, 0.5), (100, 1.5)]):
    vals = []
    for m in modules_no_overhead:
        if pct == 100:
            vals.append(df[m].max())
        else:
            vals.append(np.percentile(df[m], pct))
    
    label = f'P{pct}' if pct != 100 else 'Max'
    shade = ['#2a78d6', '#1baf7a', '#eda100', '#e34948'][j]
    ax.bar(x + offset * w, vals, w, label=label, color=shade, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([MODULE_LABELS[m] for m in modules_no_overhead])
ax.set_ylabel('Latency (μs)')
ax.set_title('Per-Module Latency Percentiles')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Latency Over Time

Does latency degrade as the system processes more ticks? A rising trend would indicate memory accumulation (growing deques, unbounded caches). A flat line means the streaming design is working correctly.

In [ ]:
# Total latency over tick count (rolling mean to smooth)
fig, ax = plt.subplots(figsize=(14, 4))

window = 100
rolling_total = df['total_us'].rolling(window).mean()
ax.plot(df['tick_id'], rolling_total, color='#2a78d6', linewidth=1)
ax.axhline(df['total_us'].mean(), color='#e34948', linewidth=0.8, linestyle='--',
           label=f'Mean: {df["total_us"].mean():.0f} μs')

ax.set_title(f'Total Latency Over Time ({window}-tick rolling mean)')
ax.set_xlabel('Tick')
ax.set_ylabel('Latency (μs)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# OFI latency over time
fig, ax = plt.subplots(figsize=(14, 4))

rolling_ofi = df['ofi_us'].rolling(window).mean()
ax.plot(df['tick_id'], rolling_ofi, color='#eb6834', linewidth=1)
ax.axhline(df['ofi_us'].mean(), color='#e34948', linewidth=0.8, linestyle='--',
           label=f'Mean: {df["ofi_us"].mean():.0f} μs')

ax.set_title(f'OFI Module Latency Over Time ({window}-tick rolling mean)')
ax.set_xlabel('Tick')
ax.set_ylabel('Latency (μs)')
ax.legend()
plt.tight_layout()
plt.show()

# Check for trend
from scipy import stats as sp_stats
slope, _, r, p, _ = sp_stats.linregress(df['tick_id'], df['ofi_us'])
print(f'OFI latency trend: slope={slope:.4f} μs/tick, r²={r**2:.4f}, p={p:.2e}')
if abs(slope) < 0.01:
    print('→ No significant trend — streaming design is working correctly')
else:
    print(f'→ Trend detected: {slope * 1000:.2f} μs per 1000 ticks')

---
## 5. Per-Symbol Comparison

Do some symbols take longer to process than others? This could happen if certain symbols trigger more anomalies or have larger OFI deques.

In [ ]:
# Per-symbol latency boxplot
SYMBOL_COLORS = {
    'RELIANCE': '#2a78d6', 'TCS': '#eb6834', 'HDFCBANK': '#1baf7a',
    'INFY': '#eda100', 'ICICIBANK': '#e87ba4',
}

fig, ax = plt.subplots(figsize=(10, 4.5))

sym_data = [df[df['symbol'] == s]['total_us'] for s in SYMBOL_COLORS]
bp = ax.boxplot(sym_data, labels=list(SYMBOL_COLORS.keys()), patch_artist=True,
                widths=0.5, medianprops={'color': TEXT_PRIMARY, 'linewidth': 1.5},
                whiskerprops={'color': TEXT_MUTED}, capprops={'color': TEXT_MUTED},
                flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.3})

for patch, color in zip(bp['boxes'], SYMBOL_COLORS.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Total Latency by Symbol')
ax.set_ylabel('Latency (μs)')
plt.tight_layout()
plt.show()

# Print per-symbol stats
print('Per-symbol mean latency (μs):')
for s in SYMBOL_COLORS:
    d = df[df['symbol'] == s]['total_us']
    print(f'  {s:<12} mean={d.mean():7.1f}  p99={d.quantile(0.99):7.1f}  max={d.max():7.1f}')

---
## 6. Bottleneck Analysis

The current implementation uses bounded queues and running statistics. The slowest module can vary by hardware, warm-up state, regression window, garbage collection, and operating-system scheduling. The analysis below therefore identifies bottlenecks from the generated timing file instead of assuming one in advance.

In [ ]:
# Rank measured modules by mean and P99 latency
module_cols = [m for m in MODULE_LABELS if m != 'overhead_us']
bottlenecks = pd.DataFrame({
    'module': [MODULE_LABELS[m] for m in module_cols],
    'mean_us': [df[m].mean() for m in module_cols],
    'p99_us': [df[m].quantile(0.99) for m in module_cols],
}).sort_values('mean_us')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(bottlenecks['module'], bottlenecks['mean_us'], color='#2a78d6', alpha=0.8)
ax.set_xlabel('Mean latency (μs)')
ax.set_title('Measured Module Bottlenecks')
plt.tight_layout()
plt.show()

print(bottlenecks.sort_values('mean_us', ascending=False).to_string(index=False))

---

## Key Findings

1. **Use the generated measurements.** Mean and percentile latency are read from `latency_summary.json`; no fixed cross-machine result is claimed.

2. **Profile all nine modules.** The timing record includes the four advanced diagnostics as well as spread, OFI, VWAP, volume, and anomaly detection.

3. **Check trends rather than assuming stability.** The notebook plots rolling latency and reports a regression slope for the profiled run.

4. **Compare symbols.** Symbol-level distributions reveal whether data characteristics or warm-up stages create hotspots.

5. **Treat outliers separately.** Garbage collection and operating-system scheduling can affect maxima; report P50/P95/P99 with the machine and run configuration.

---
*Latency Profiler Report · Real-Time Market Microstructure Analyzer*  
*Aarya Parekh · IIT Bombay · August 2026*